# Step 13 — A pretrained model at CIFAR-10 scale (EDM, DDPM++, 56M parameters)

Responds to review point 8.

**Setting.** NVIDIA's public EDM checkpoint `edm-cifar10-32x32-uncond-vp.pkl` (Karras et al. 2022). Because the model is unconditional, the exact reference along any direction is the **full CIFAR-10 training set** projected onto that direction. Along a *pairwise* LDA direction the other eight classes pile up in the middle, so the marginal is not a two-branch transition; the right collective variables are directions along which the whole distribution is two lumps: **one class vs the rest** (10 directions, weight 1:9) and **vehicles vs animals** (airplane, automobile, ship, truck vs the six animal classes; weight 4:6).

**A (CPU).** Exact census along these 11 directions (bisected $\tau_{\rm bif}$, cusp fit), and, as a data-only side result comparable to MNIST, the balanced-pair census on the 45 pair subsets.

**B (GPU).** Network drift diagrams along the vehicles/animals direction and three one-vs-rest directions: bisected $\tau_{\rm bif}$ and $h_{\rm net}$ vs exact. ~10–15 min per direction on a T4.

Internet on (clones `NVlabs/edm` and downloads the ~200 MB pickle). GPU on.

In [ ]:
import os, sys, subprocess, pickle, numpy as np, torch, math, time
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
from scipy.signal import fftconvolve
from scipy.stats import theilslopes
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev); rng = np.random.default_rng(0); plt.rcParams['figure.dpi'] = 130
if not os.path.exists('/kaggle/working/edm'): subprocess.run(['git', 'clone', '-q', 'https://github.com/NVlabs/edm.git', '/kaggle/working/edm'], check=True)
sys.path.insert(0, '/kaggle/working/edm'); import dnnlib
d = datasets.CIFAR10('/kaggle/working/cifar', train=True, download=True)
X = (np.asarray(d.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2).reshape(-1, 3072); Y = np.asarray(d.targets); print(X.shape)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]
def smooth_hist(m0, tau):
    h, _ = np.histogram(m0, bins=np.append(GRID-DG/2, GRID[-1]+DG/2), density=True); k = np.exp(-GRID**2/(2*tau)); k /= k.sum()
    return np.maximum(fftconvolve(h, k, mode='same'), 1e-300)
def exact_drift(m0, tau): p = smooth_hist(m0, tau); return GRID, np.gradient(np.log(p), GRID), p
def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary_points(p):
    d_ = np.gradient(np.log(p), GRID); keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d_[:-1]) != np.sign(d_[1:])) & keep[:-1] & keep[1:])[0]
    return _merge([(float(GRID[i]-d_[i]*DG/(d_[i+1]-d_[i])) if d_[i+1] != d_[i] else float(GRID[i]), 'max' if d_[i] > 0 else 'min') for i in idx])
def n_max(pts): return sum(k == 'max' for _, k in pts)
def birth_bisect(get_pts, taus=np.geomspace(0.05, 3.0, 60), iters=10):
    ts = sorted(taus, reverse=True); hi = None
    for tau in ts:
        pts = get_pts(tau)
        if n_max(pts) >= 2 and hi is not None: lo = tau; break
        if n_max(pts) == 1: hi = tau
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    pm = [p for p, k in get_pts(hi) if k == 'max']; prev = pm[0] if pm else 0.0; pl = get_pts(lo)
    mx = sorted(p for p, k in pl if k == 'max'); mn = [p for p, k in pl if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev))
def cusp_fit(m, f, w, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(m) < window) & (w > 0); c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok]))
    b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b, clamped=(-c3 <= b_min+1e-12))
def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(Xa.shape[1]), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)

## A. Exact census of the full CIFAR-10 distribution along the 45 LDA directions (CPU)

In [ ]:
VEH = [0, 1, 8, 9]; ANI = [2, 3, 4, 5, 6, 7]
def analyse_direction(Xa, Xb, Xfull, label):
    u, mid, half = lda_direction(Xa, Xb); m0 = (Xfull - mid) @ u / half
    bj = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t)))
    if bj is None: print(f"{label:>26}: never bimodal"); return None
    f = cusp_fit(*exact_drift(m0, bj[0])); r = dict(label=label, tau=bj[0], jump=bj[1], u=u, mid=mid, half=half, m0=m0, **f)
    print(f"{label:>26}: tau_bif={bj[0]:.3f} jump={bj[1]:.2f} h={f['h']:+.4f} a={f['a']:+.3f} b={f['b']:.3f} half={half:.2f}{'  [b clamped]' if f['clamped'] else ''}"); return r

print("== Directions along which the FULL distribution is two lumps (what the unconditional model sees) ==")
DIRS = []
r = analyse_direction(X[np.isin(Y, VEH)], X[np.isin(Y, ANI)], X, 'vehicles vs animals');  DIRS.append(r)
for c in range(10):
    r = analyse_direction(X[Y == c], X[Y != c], X, f'{CLASSES[c]} vs rest')
    if r: DIRS.append(r)

print("\n== Data-only balanced-pair census on pair subsets (comparable to MNIST; NOT what the unconditional model sees) ==")
PAIRS = []
for a_, b_ in combinations(range(10), 2):
    Xa, Xb = X[Y == a_], X[Y == b_]; u, mid, half = lda_direction(Xa, Xb); m0 = np.concatenate([(Xa-mid) @ u, (Xb-mid) @ u]) / half
    bj = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t)))
    if bj is None: continue
    f = cusp_fit(*exact_drift(m0, bj[0])); PAIRS.append(dict(pair=(a_, b_), tau=bj[0], jump=bj[1], m0=m0, **f))
fl = []
for r in PAIRS[:12]:
    p = r['m0']; idx = rng.permutation(len(p)); k = len(p)//2; ms = np.concatenate([p[idx[:k]], -p[idx[k:2*k]]])
    bj = birth_bisect(lambda t: stationary_points(smooth_hist(ms, t)))
    if bj: fl.append(abs(cusp_fit(*exact_drift(ms, bj[0]))['h']))
FLOOR = float(np.percentile(fl, 95)) if fl else 0.005
H = np.array([abs(r['h']) for r in PAIRS]); ok = (H > FLOOR) & np.array([not r['clamped'] and r['tau'] >= 0.45 for r in PAIRS])
print(f"CIFAR-10 balanced pairs: {len(PAIRS)}/45 bimodal; |h| median {np.median(H):.4f}, max {H.max():.3f}; floor {FLOOR:.4f}; above floor and non-degenerate: {ok.sum()}")
if ok.sum() >= 5:
    x = np.log(H[ok]/(2*np.array([r['b'] for r in PAIRS])[ok])); y = np.log(np.array([r['jump'] for r in PAIRS])[ok]); sl, ic = np.polyfit(x, y, 1); ts = theilslopes(y, x)
    print(f"cube-root test: OLS p={sl:.3f} C={np.exp(ic):.2f}; Theil-Sen p={ts[0]:.3f} [{ts[2]:.3f}, {ts[3]:.3f}]  (theory 1/3, 3)")
plt.figure(figsize=(5, 3.4)); plt.hist(H, bins=25); plt.axvline(FLOOR, color='k', ls='--', label='floor'); plt.xlabel('|h| over 45 balanced CIFAR-10 pairs (data only)'); plt.legend(); plt.tight_layout(); plt.show()

## B. Pretrained network fidelity along selected directions (GPU)

In [ ]:
URL = 'https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl'
with dnnlib.util.open_url(URL) as f: net = pickle.load(f)['ema'].to(dev).eval()
print('loaded EDM CIFAR-10 unconditional; sigma range', net.sigma_min, net.sigma_max)

def zero_crossings(cen, mean, cnt, min_count=60):
    okk = cnt >= min_count; c, f = cen[okk], mean[okk]; out = []
    for i in range(len(c)-1):
        if np.sign(f[i]) != np.sign(f[i+1]) and f[i] != 0: out.append((float(c[i]-f[i]*(c[i+1]-c[i])/(f[i+1]-f[i])), 'max' if f[i] > 0 else 'min'))
    return _merge(out, merge=0.05)
@torch.no_grad()
def net_binned(u, mid, half, tau, n=8000, nbins=50, bs=250):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); sigma = float(np.clip(half*np.sqrt(tau), net.sigma_min, net.sigma_max))
    x0 = torch.from_numpy(X[rng.choice(len(X), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0); D = []
    for xb in xt.split(bs): D.append(net(xb.view(-1, 3, 32, 32), torch.full((len(xb),), sigma, device=dev)).view(len(xb), -1))
    D = torch.cat(D); m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
    lo, hi = np.quantile(m, [0.005, 0.995]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
    cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins); k = np.exp(-0.5*(np.arange(-5, 6)/1.5)**2)
    return 0.5*(edges[:-1]+edges[1:]), np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9), cnt

byname = {r['label']: r for r in DIRS if r}; pick = [byname[k] for k in ['vehicles vs animals', 'airplane vs rest', 'frog vs rest', 'truck vs rest'] if k in byname]
def drift_rms(r, tau):
    """Relative RMS difference between network and exact drift on |m|<1.5 at this tau (cubic-free fidelity measure)."""
    cen, gnet, cnt = net_binned(r['u'], r['mid'], r['half'], tau); g, gex, p = exact_drift(r['m0'], tau)
    okk = (np.abs(cen) < 1.5) & (cnt >= 60); ge = np.interp(cen[okk], g, gex); return float(np.sqrt(np.mean((gnet[okk]-ge)**2))/np.sqrt(np.mean(ge**2)))
print(f"{'direction':>24} {'exact tau_bif':>14} {'net tau_bif':>12} {'exact h':>9} {'net h':>8} {'drift RMS err @tau_bif':>24} {'@2 tau_bif':>11}")
RES = []
for r in pick:
    t0 = time.time()
    bj = birth_bisect(lambda t: zero_crossings(*net_binned(r['u'], r['mid'], r['half'], t)), taus=np.geomspace(0.1, 3.0, 25), iters=6)
    taus = np.geomspace(0.4, 1.5, 6); hn = np.mean([cusp_fit(*net_binned(r['u'], r['mid'], r['half'], t))['h'] for t in taus]); hx = np.mean([cusp_fit(*exact_drift(r['m0'], t))['h'] for t in taus])
    e1, e2 = drift_rms(r, r['tau']), drift_rms(r, 2*r['tau'])
    RES.append((r['label'], r['tau'], bj, hx, hn, e1, e2)); print(f"{r['label']:>24} {r['tau']:14.3f} {(bj[0] if bj else float('nan')):12.3f} {hx:+9.3f} {hn:+8.3f} {e1:24.3f} {e2:11.3f}   ({time.time()-t0:.0f}s)")
print("(h values are only meaningful for the nearly balanced vehicles/animals direction; for 1:9 directions compare tau_bif and the drift RMS error.)")

fig, axes = plt.subplots(1, len(pick), figsize=(3.4*len(pick), 3.4))
for ax, r in zip(axes, pick):
    for tau in np.geomspace(0.1, 3.0, 25):
        for p, k in stationary_points(smooth_hist(r['m0'], tau)): ax.plot(tau, p, '_', ms=8, color='k', alpha=0.4)
        for p, k in zero_crossings(*net_binned(r['u'], r['mid'], r['half'], tau, n=4000)): ax.plot(tau, p, 'o', ms=3, color='C0' if k == 'max' else 'C3')
    ax.set_xscale('log'); ax.set_xlabel('tau'); ax.set_title(f"{r['label']}  h={r['h']:+.3f}", fontsize=9); ax.axhline(0, color='k', lw=0.4)
plt.tight_layout(); plt.show()

## What to report back
**A.** The 11-direction table (these are the transitions the unconditional model must reproduce) and the balanced-pair census line and histogram (data-only, comparable to MNIST).
**B.** The table and the five diagrams: does a 56M-parameter pretrained model reproduce the exact transition (τ_bif ratio near 1, h_net ≈ h_exact), or does it show the same delay and excess asymmetry as our small models?